# Assemble Datasets for SFT Training

## Setup and Imports

In [2]:
import pandas as pd
import os
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs, MACCSkeys, rdFMCS, Draw
from tqdm.auto import tqdm  # for notebooks

tqdm.pandas()

pd.set_option('display.max_columns', 1000, 'display.width', 2000, 'display.max_colwidth', 100)

data_dir = os.path.join(os.getcwd(), '..', 'data')

In [23]:
pred_df = pd.read_csv(os.path.join(data_dir, "protac_splitter_predictions_sample.csv"))
pred_df.columns

Index(['model', 'text', 'labels', 'dataset', 'generated_text-0', 'generated_text-1', 'generated_text-2', 'generated_text-3', 'generated_text-4'], dtype='object')

In [24]:
print(pred_df.shape)

(21456, 9)


In [25]:
models = list(pred_df['model'].unique())
for m in models:
    print(m)
print('')
datasets = [s.replace('ailab-bio/PROTAC-Splitter_', '').replace('perc', '%') for s in models]
models2datasets = {m: d for m, d in zip(models, datasets)}
datasets2models = {d: m for m, d in zip(models, datasets)}
acronyms = {
    'ailab-bio/PROTAC-Splitter': 'PS',
    'untied': 'U',
    # '80-20-split': '80/20',
    'shuffled-labels-order-50perc': 'SLO',
    'random-labels-smiles-50perc': 'RLS',
}
def get_acronym(model: str) -> str:
    return '-'.join([acronyms[m] for m in model.split('_') if m in acronyms])
models2acronyms = {m: get_acronym(m) for m in models}
acronyms2models = {a: m for m, a in models2acronyms.items()}
for a, m in acronyms2models.items():
    print(f"{m} -> {a}")

ailab-bio/PROTAC-Splitter_80-20-split_shuffled-labels-order-50perc_random-labels-smiles-50perc
ailab-bio/PROTAC-Splitter_80-20-split_random-labels-smiles-50perc
ailab-bio/PROTAC-Splitter_80-20-split_shuffled-labels-order-50perc
ailab-bio/PROTAC-Splitter_untied_80-20-split_shuffled-labels-order-50perc
ailab-bio/PROTAC-Splitter_untied_80-20-split_shuffled-labels-order-50perc_random-labels-smiles-50perc
ailab-bio/PROTAC-Splitter_untied_80-20-split_random-labels-smiles-50perc

ailab-bio/PROTAC-Splitter_80-20-split_shuffled-labels-order-50perc_random-labels-smiles-50perc -> PS-SLO-RLS
ailab-bio/PROTAC-Splitter_80-20-split_random-labels-smiles-50perc -> PS-RLS
ailab-bio/PROTAC-Splitter_80-20-split_shuffled-labels-order-50perc -> PS-SLO
ailab-bio/PROTAC-Splitter_untied_80-20-split_shuffled-labels-order-50perc -> PS-U-SLO
ailab-bio/PROTAC-Splitter_untied_80-20-split_shuffled-labels-order-50perc_random-labels-smiles-50perc -> PS-U-SLO-RLS
ailab-bio/PROTAC-Splitter_untied_80-20-split_random-labe

In [26]:
from rdkit import RDLogger

def is_valid_smiles(smiles: str) -> bool:
    mol = Chem.MolFromSmiles(smiles)
    return True if mol is not None else False

def has_three_substructures(smiles: str) -> bool:
    return smiles.count(".") == 2

def has_all_attachment_points(smiles: str) -> bool:
    return smiles.count("[*:1]") == 2 and smiles.count("[*:2]") == 2

def same_atom_counts_and_types(smiles1, smiles2, get_atoms_diff=False):
    """
    Check if two molecules have the same number and types of atoms.

    Args:
    smiles1 (str): SMILES notation for the first molecule.
    smiles2 (str): SMILES notation for the second molecule.

    Returns:
    bool: True if the molecules have the same atom counts and types, False otherwise.
    """
    mol1 = Chem.MolFromSmiles(smiles1)
    mol2 = Chem.MolFromSmiles(smiles2)
    if mol1 is None or mol2 is None:
        if get_atoms_diff:
            return float("nan")
            # raise ValueError("Invalid SMILES notation provided for one or both molecules.")
        else:
            return False
    num_atoms1 = Chem.rdMolDescriptors.CalcNumHeavyAtoms(mol1)
    num_atoms2 = Chem.rdMolDescriptors.CalcNumHeavyAtoms(mol2)
    if get_atoms_diff:
        return abs(num_atoms1 - num_atoms2)
        # tmp = {}
        # for atom in atom_counts1.keys():
        #     tmp[atom] = int(abs(atom_counts1.get(atom, 0) - atom_counts2.get(atom, 0)))
        # for atom in atom_counts2.keys():
        #     tmp[atom] = int(abs(atom_counts1.get(atom, 0) - atom_counts2.get(atom, 0)))
        # return tmp # abs(atom_counts1.get('O', 0) - atom_counts2.get('O', 0))
    else:
        atom_counts1, atom_counts2 = {}, {}
        for atom in mol1.GetAtoms():
            if '*' not in atom.GetSmarts():
                atom_counts1[atom.GetSymbol()] = atom_counts1.get(atom.GetSymbol(), 0) + 1
        for atom in mol2.GetAtoms():
            if '*' not in atom.GetSmarts():
                atom_counts2[atom.GetSymbol()] = atom_counts2.get(atom.GetSymbol(), 0) + 1
        return (atom_counts1 == atom_counts2) & (num_atoms1 == num_atoms2)


def get_checks(row):
    valid_prediction = False
    for i in range(5):
        row[f'valid_smiles-{i}'] = is_valid_smiles(row[f'generated_text-{i}'])
        row[f'has_three_substructures-{i}'] = has_three_substructures(row[f'generated_text-{i}'])
        row[f'has_all_attachment_points-{i}'] = has_all_attachment_points(row[f'generated_text-{i}'])
        row[f'same_atom_counts_and_types-{i}'] = same_atom_counts_and_types(row['text'], row[f'generated_text-{i}'])
        row[f'atom_difference_counts-{i}'] = same_atom_counts_and_types(row['text'], row[f'generated_text-{i}'], get_atoms_diff=True)
        # If any of the above checks fail, the prediction is invalid
        row[f'valid_prediction-{i}'] = row[f'valid_smiles-{i}'] & row[f'has_three_substructures-{i}'] & row[f'has_all_attachment_points-{i}'] & row[f'same_atom_counts_and_types-{i}']
        valid_prediction |= row[f'valid_prediction-{i}']
    row['valid_prediction'] = valid_prediction
    return row

if os.path.exists(os.path.join(data_dir, "'protac_splitter_predictions_sample_evaluated.csv")):
    pred_df = pd.read_csv(os.path.join(data_dir, "'protac_splitter_predictions_sample_evaluated.csv"))
else:
    RDLogger.DisableLog("rdApp.*")
    tqdm.pandas(desc='Get checks')
    pred_df = pred_df.progress_apply(get_checks, axis=1)
    pred_df.to_csv(os.path.join(data_dir, 'protac_splitter_predictions_sample_evaluated.csv'), index=False)
    display(pred_df.head())

Get checks:   0%|          | 0/21456 [00:00<?, ?it/s]

,model,text,labels,dataset,generated_text-0,generated_text-1,generated_text-2,generated_text-3,generated_text-4,valid_smiles-0,has_three_substructures-0,has_all_attachment_points-0,same_atom_counts_and_types-0,atom_difference_counts-0,valid_prediction-0,valid_smiles-1,has_three_substructures-1,has_all_attachment_points-1,same_atom_counts_and_types-1,atom_difference_counts-1,valid_prediction-1,valid_smiles-2,has_three_substructures-2,has_all_attachment_points-2,same_atom_counts_and_types-2,atom_difference_counts-2,valid_prediction-2,valid_smiles-3,has_three_substructures-3,has_all_attachment_points-3,same_atom_counts_and_types-3,atom_difference_counts-3,valid_prediction-3,valid_smiles-4,has_three_substructures-4,has_all_attachment_points-4,same_atom_counts_and_types-4,atom_difference_counts-4,valid_prediction-4,valid_prediction
0,ailab-bio/PROTAC-Splitter_80-20-split_shuffled-labels-order-50perc_random-labels-smiles-50perc,C=CC(=O)Nc1cccc(-n2c(=O)cc(C)c3cnc(Nc4ccc(N5CCN(CCCCCCCCCCCC(=O)NC(C(=O)N6CC(O)CC6C(=O)NCc6ccc(-...,[*:2]NC(C(=O)N1CC(O)CC1C(=O)NCc1ccc(-c2scnc2C)cc1)C(C)(C)C.[*:2]C(=O)CCCCCCCCCC[*:1].[*:1]CN1CCN...,train,[*:1]CN1CCN(c2ccc(Nc3ncc4c(C)cc(=O)n(-c5cccc(NC(=O)C=C)c5)c4n3)c(OC)c2)CC1.[*:2]NC(C(=O)N1CC(O)C...,[*:1]CN1CCN(c2ccc(Nc3ncc4c(C)cc(=O)n(-c5cccc(NC(=O)C=C)c5)c4n3)c(OC)c2)CC1.[*:2]NC(C(=O)N1CC(O)C...,[*:1]CN1CCN(c2ccc(Nc3ncc4c(C)cc(=O)n(-c5cccc(NC(=O)C=C)c5)c4n3)c(OC)c2)CC1.[*:2]NC(C(=O)N1CC(O)C...,[*:1]CN1CCN(c2ccc(Nc3ncc4c(C)cc(=O)n(-c5cccc(NC(=O)C=C)c5)c4n3)c(OC)c2)CC1.[*:2]NC(C(=O)N1CC(O)C...,[*:1]CN1CCN(c2ccc(Nc3ncc4c(C)cc(=O)n(-c5cccc(NC(=O)C=C)c5)c4n3)c(OC)c2)CC1.[*:2]NC(C(=O)N1CC(O)C...,True,True,True,False,5.0,False,True,True,True,False,5.0,False,True,True,True,False,5.0,False,True,True,True,False,5.0,False,True,True,True,False,1.0,False,False
1,ailab-bio/PROTAC-Splitter_80-20-split_shuffled-labels-order-50perc_random-labels-smiles-50perc,C=CC(=O)Nc1cccc(-n2c(=O)cc(C)c3cnc(Nc4ccc(N5CCN(CCCCCCCC(=O)NC(C(=O)N6CC(O)CC6C(=O)NCc6ccc(-c7sc...,[*:1]CN1CCN(c2ccc(Nc3ncc4c(C)cc(=O)n(-c5cccc(NC(=O)C=C)c5)c4n3)c(OC)c2)CC1.[*:2]NC(C(=O)N1CC(O)C...,train,[*:1]CN1CCN(c2ccc(Nc3ncc4c(C)cc(=O)n(-c5cccc(NC(=O)C=C)c5)c4n3)c(OC)c2)CC1.[*:2]NC(C(=O)N1CC(O)C...,[*:1]CN1CCN(c2ccc(Nc3ncc4c(C)cc(=O)n(-c5cccc(NC(=O)C=C)c5)c4n3)c(OC)c2)CC1.[*:2]NC(C(=O)N1CC(O)C...,[*:1]CN1CCN(c2ccc(Nc3ncc4c(C)cc(=O)n(-c5cccc(NC(=O)C=C)c5)c4n3)c(OC)c2)CC1.[*:2]NC(C(=O)N1CC(O)C...,[*:1]CN1CCN(c2ccc(Nc3ncc4c(C)cc(=O)n(-c5cccc(NC(=O)C=C)c5)c4n3)c(OC)c2)CC1.[*:2]NC(C(=O)N1CC(O)C...,[*:1]CN1CCN(c2ccc(Nc3ncc4c(C)cc(=O)n(-c5cccc(NC(=O)C=C)c5)c4n3)c(OC)c2)CC1.[*:2]NC(C(=O)N1CC(O)C...,True,True,True,False,1.0,False,True,True,True,False,1.0,False,True,True,True,False,1.0,False,True,True,True,False,4.0,False,True,True,True,False,4.0,False,False
2,ailab-bio/PROTAC-Splitter_80-20-split_shuffled-labels-order-50perc_random-labels-smiles-50perc,C=CC(=O)Nc1cccc(-n2c(=O)cc(C)c3cnc(Nc4ccc(N5CCN(CCCCCC(=O)NC(C(=O)N6CC(O)CC6C(=O)NCc6ccc(-c7scnc...,[*:1]CN1CCN(c2ccc(Nc3ncc4c(C)cc(=O)n(-c5cccc(NC(=O)C=C)c5)c4n3)c(OC)c2)CC1.[*:2]NC(C(=O)N1CC(O)C...,train,[*:1]CN1CCN(c2ccc(Nc3ncc4c(C)cc(=O)n(-c5cccc(NC(=O)C=C)c5)c4n3)c(OC)c2)CC1.[*:2]NC(C(=O)N1CC(O)C...,[*:1]CN1CCN(c2ccc(Nc3ncc4c(C)cc(=O)n(-c5cccc(NC(=O)C=C)c5)c4n3)c(OC)c2)CC1.[*:2]NC(C(=O)N1CC(O)C...,[*:1]CN1CCN(c2ccc(Nc3ncc4c(C)cc(=O)n(-c5cccc(NC(=O)C=C)c5)c4n3)c(OC)c2)CC1.[*:2]NC(C(=O)N1CC(O)C...,[*:1]CN1CCN(c2ccc(Nc3ncc4c(C)cc(=O)n(-c5cccc(NC(=O)C=C)c5)c4n3)c(OC)c2)CC1.[*:2]NC(C(=O)N1CC(O)C...,[*:1]CN1CCN(c2ccc(Nc3ncc4c(C)cc(=O)n(-c5cccc(NC(=O)C=C)c5)c4n3)c(OC)c2)CC1.[*:2]NC(C(=O)N1CC(O)C...,True,True,True,False,1.0,False,True,True,True,False,1.0,False,True,True,True,False,1.0,False,True,True,True,False,2.0,False,True,True,True,False,2.0,False,False
3,ailab-bio/PROTAC-Splitter_80-20-split_shuffled-labels-order-50perc_random-labels-smiles-50perc,C=CC(=O)Nc1cccc(-n2c(=O)cc(C)c3cnc(Nc4ccc(N5CCN(CCCC(=O)NC(C(=O)N6CC(O)CC6C(=O)NCc6ccc(-c7scnc7C...,[*:1]CN1CCN(c2ccc(Nc3ncc4c(C)cc(=O)n(-c5cccc(NC(=O)C=C)c5)c4n3)c(OC)c2)C

In [30]:
len_df = len(pred_df[pred_df['dataset'] == 'validation'])
len_valid = ((pred_df['valid_prediction'] == True) & (pred_df['dataset'] == 'validation')).sum()
print(f"Valid predictions: {len_valid} / {len_df} ({len_valid / len_df * 100:.2f}%)")

Valid predictions: 265 / 2022 (13.11%)


In [46]:
# tmp = pred_df[pred_df['dataset'] != 'validation']
tmp = pred_df[pred_df['dataset'] == 'train']
dpo_dataset = []
for i, row in tmp.iterrows():
    for rejected_idx in range(5):
        if row[f'valid_prediction-{rejected_idx}'] == False:
            # Check if row['labels'] is not NaN
            if row['labels'] == row['labels']:
                dpo_dataset.append({
                    'prompt': row['text'],
                    'chosen': row['labels'],
                    'rejected': row[f'generated_text-{rejected_idx}'],
                })
            for chosen_idx in range(5):
                if row[f'valid_prediction-{chosen_idx}'] == True:
                    dpo_dataset.append({
                        'prompt': row['text'],
                        'chosen': row[f'generated_text-{chosen_idx}'],
                        'rejected': row[f'generated_text-{rejected_idx}'],
                    })
dpo_dataset = pd.DataFrame(dpo_dataset).drop_duplicates()
dpo_dataset

,prompt,chosen,rejected
0,C=CC(=O)Nc1cccc(-n2c(=O)cc(C)c3cnc(Nc4ccc(N5CCN(CCCCCCCCCCCC(=O)NC(C(=O)N6CC(O)CC6C(=O)NCc6ccc(-...,[*:2]NC(C(=O)N1CC(O)CC1C(=O)NCc1ccc(-c2scnc2C)cc1)C(C)(C)C.[*:2]C(=O)CCCCCCCCCC[*:1].[*:1]CN1CCN...,[*:1]CN1CCN(c2ccc(Nc3ncc4c(C)cc(=O)n(-c5cccc(NC(=O)C=C)c5)c4n3)c(OC)c2)CC1.[*:2]NC(C(=O)N1CC(O)C...
4,C=CC(=O)Nc1cccc(-n2c(=O)cc(C)c3cnc(Nc4ccc(N5CCN(CCCCCCCCCCCC(=O)NC(C(=O)N6CC(O)CC6C(=O)NCc6ccc(-...,[*:2]NC(C(=O)N1CC(O)CC1C(=O)NCc1ccc(-c2scnc2C)cc1)C(C)(C)C.[*:2]C(=O)CCCCCCCCCC[*:1].[*:1]CN1CCN...,[*:1]CN1CCN(c2ccc(Nc3ncc4c(C)cc(=O)n(-c5cccc(NC(=O)C=C)c5)c4n3)c(OC)c2)CC1.[*:2]NC(C(=O)N1CC(O)C...
5,C=CC(=O)Nc1cccc(-n2c(=O)cc(C)c3cnc(Nc4ccc(N5CCN(CCCCCCCC(=O)NC(C(=O)N6CC(O)CC6C(=O)NCc6ccc(-c7sc...,[*:1]CN1CCN(c2ccc(Nc3ncc4c(C)cc(=O)n(-c5cccc(NC(=O)C=C)c5)c4n3)c(OC)c2)CC1.[*:2]NC(C(=O)N1CC(O)C...,[*:1]CN1CCN(c2ccc(Nc3ncc4c(C)cc(=O)n(-c5cccc(NC(=O)C=C)c5)c4n3)c(OC)c2)CC1.[*:2]NC(C(=O)N1CC(O)C...
8,C=CC(=O)Nc1cccc(-n2c(=O)cc(C)c3cnc(Nc4ccc(N5CCN(CCCCCCCC(=O)NC(C(=O)N6CC(O)CC6C(=O)NCc6ccc(-c7sc...,[*:1]CN1CCN(c2ccc(Nc3ncc4c(C)cc(=O)n(-c5cccc(NC(=O)C=C)c5)c4n3)c(OC)c2)CC1.[*:2]NC(C(=O)N1CC(O)C...,[*:1]CN1CCN(c2ccc(Nc3ncc4c(C)cc(=O)n(-c5cccc(NC(=O)C=C)c5)c4n3)c(OC)c2)CC1.[*:2]NC(C(=O)N1CC(O)C...
10,C=CC(=O)Nc1cccc(-n2c(=O)cc(C)c3cnc(Nc4ccc(N5CCN(CCCCCC(=O)NC(C(=O)N6CC(O)CC6C(=O)NCc6ccc(-c7scnc...,[*:1]CN1CCN(c2ccc(Nc3ncc4c(C)cc(=O)n(-c5cccc(NC(=O)C=C)c5)c4n3)c(OC)c2)CC1.[*:2]NC(C(=O)N1CC(O)C...,[*:1]CN1CCN(c2ccc(Nc3ncc4c(C)cc(=O)n(-c5cccc(NC(=O)C=C)c5)c4n3)c(OC)c2)CC1.[*:2]NC(C(=O)N1CC(O)C...
...,...,...,...
37296,Cc1ncsc1-c1ccc(C(C)NC(=O)C2CC(O)CN2C(=O)C(NC(=O)CCCCCCCCCCN2CCN(CC(=O)Nc3cccc(Sc4ncc(N5CCC(C)(N)...,[*:1]Nc1cccc(Sc2ncc(N3CCC(C)(N)CC3)nc2N)c1Cl.[*:2]CC(=O)NC(C(=O)N1CC(O)CC1C(=O)NC(C)c1ccc(-c2scn...,[*:1]Nc1cccc(Sc2ncc(N3CCC(C)(N)CC3)nc2N)c1Cl.[*:2]CC(=O)NC(C(=O)N1CC(O)CC1C(=O)NC(C)c1ccc(-c2scn...
37308,Cc1ncsc1-c1ccc(C(C)NC(=O)C2CC(O)CN2C(=O)C(NC(=O)CCCCCCCCCCCN2CCN(CC(=O)Nc3cccc(Sc4ncc(N5CCC(C)(N...,[*:1]Nc1cccc(Sc2ncc(N3CCC(C)(N)CC3)nc2N)c1Cl.[*:2]CC(=O)NC(C(=O)N1CC(O)CC1C(=O)NC(C)c1ccc(-c2scn...,[*:1]Nc1cccc(Sc2ncc(N3CCC(C)(N)CC3)nc2N)c1Cl.[*:2]CC(=O)NC(C(=O)N1CC(O)CC1C(=O)NC(C)c1ccc(-c2scn...
37310,Cc1ncsc1-c1ccc(C(C)NC(=O)C2CC(O)CN2C(=O)C(NC(=O)CCCCCCCCCN2CCN(C(=O)CCC(=O)Nc3cccc(Sc4ncc(N5CCC(...,[*:1]Nc1cccc(Sc2ncc(N3CCC(C)(N)CC3)nc2N)c1Cl.[*:2]CC(=O)NC(C(=O)N1CC(O)CC1C(=O)NC(C)c1ccc(-c2scn...,[*:1]CC(=O)Nc1cccc(Sc2ncc(N3CCC(C)(N)CC3)nc2N)c1Cl.[*:2]CC(=O)NC(C(=O)N1CC(O)CC1C(=O)NC(C)c1ccc(...
37324,C=C(F)C(=O)N1CCN(c2nc(OCC3CCCN3CCCOCCC(=O)NC(C(=O)N3CC(O)CC3C(=O)NCc3ccc(-c4scnc4C)cc3)C(C)(C)C)...,[*:1]CN1CCCC1COc1nc2c(c(N3CCN(C(=O)C(=C)F)C(CC#N)C3)n1)CCN(c1cccc3cccc(Cl)c13)C2.[*:2]CC(=O)NC(C...,N(C(=O)C1N(C(=O)C(N[*:2])C(C)(C)C)CC(C1)O)Cc1ccc(cc1)-c1c(ncs1)C.O(CCOC[*:1])CCC(=O)[*:2].c1cc(C...


In [49]:
from datasets import Dataset

dataset = Dataset.from_pandas(dpo_dataset, preserve_index=False)
dataset_name = 'ailab-bio/PROTAC-Substructures-DPO'
dataset.push_to_hub(dataset_name, private=True)

Pushing dataset shards to the dataset hub:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/7 [00:00<?, ?ba/s]

Upload 1 LFS files:   0%|          | 0/1 [00:00<?, ?it/s]